###Run Shared Libraries

In [0]:
%run ../Misc/sharedlibraries

In [0]:
UpdatedDateTime = datetime.datetime.now()
Entity = "factpurchaseorder"

In [0]:
purchaseorderDf = spark.table("bronze.purchaseorder")
dimcostcenterDf = spark.table("silver.dimcostcenter")
dimcurrencyDf = spark.table("silver.dimcurrency")

In [0]:
purchaseorderDf.printSchema()
dimcostcenterDf.printSchema()
dimcurrencyDf.printSchema()

###Build Fact Purchase Order table

In [0]:
po = purchaseorderDf.alias("po")
cc = dimcostcenterDf.alias("cc")
cur = dimcurrencyDf.alias("cur")

factpurchaseorderDf = po.filter(
        F.col("po.RecordId").isNotNull()
    ).join(
        cc,
        F.col("po.CostCenter") == F.col("cc.CostCenterNumber").cast("string"),
        "left"
    ).join(
        cur,
        F.col("po.currencycode") == F.col("cur.CurrencyCode"),
        "left"
    ).select(
        F.col("po.PoNumber"),
        F.col("po.LineItem"),
        F.col("po.VendId").alias("VendorKey"),

        F.when(
            F.col("po.LastProcessedChange_DateTime").isNull(),
            F.lit("1900-01-01")
        ).otherwise(
            F.col("po.LastProcessedChange_DateTime")
        ).cast("timestamp").alias("LastProcessedChange_DateTime"),

        F.from_utc_timestamp(
            F.col("po.DataLakeModified_DateTime"),
            "CST"
        ).alias("DataLakeModified_DateTime"),

        F.col("po.Qty").cast("long").alias("Qty"),
        F.col("po.PurchasePrice").cast("double").alias("PurchasePrice"),
        F.col("po.TotalOrder").cast("double").alias("TotalOrder"),

        F.col("po.CostCenter").alias("CostCenterKey"),

        F.coalesce(
            F.col("cc.Vat").cast("double"),
            F.lit(0.0)
        ).alias("VatAmount"),

        F.round(
            F.col("po.TotalOrder").cast("double")
            + (
                F.col("po.TotalOrder").cast("double")
                * F.coalesce(F.col("cc.Vat").cast("double"), F.lit(0.0))
            ),
            4
        ).alias("TotalAmount"),

        F.col("po.ExchangeRate").cast("double").alias("ExchangeRate"),
        F.col("po.Itemkey").alias("ItemKey"),
        F.col("cur.CurrencyId").alias("CurrencyKey"),

        F.from_utc_timestamp(F.col("po.OrderDate"), "CST").alias("OrderDate"),
        F.from_utc_timestamp(F.col("po.ShipDate"), "CST").alias("ShipDate"),
        F.from_utc_timestamp(F.col("po.DeliveredDate"), "CST").alias("DeliveredDate"),

        F.date_format(
            F.from_utc_timestamp(F.col("po.OrderDate"), "CST"),
            "yyyyMMdd"
        ).cast("int").alias("OrderDateKey"),

        F.date_format(
            F.from_utc_timestamp(F.col("po.ShipDate"), "CST"),
            "yyyyMMdd"
        ).cast("int").alias("ShipDateKey"),

        F.date_format(
            F.from_utc_timestamp(F.col("po.DeliveredDate"), "CST"),
            "yyyyMMdd"
        ).cast("int").alias("DeliveredDateKey"),

        F.col("po.TrackingNumber"),
        F.col("po.Batchid").alias("BatchId"),
        F.col("po.CreatedBy"),
        F.col("po.RecordId").alias("PurchaseOrderRecordId"),
        F.col("po.CategoryId").alias("CategoryKey")
    ).withColumn(
        "UpdatedDateTime",
        F.lit(UpdatedDateTime)
    ).withColumn(
        "PurchaseOrderHashKey",
        F.xxhash64("PurchaseOrderRecordId")
    )

display(factpurchaseorderDf)

###Final dataframe

In [0]:
df_final = factpurchaseorderDf

## Write to Silver Schema

In [0]:
saveDeltaTableToCatalog(df_final,"silver",Entity)

###Validate Silver Fact Table

In [0]:
display(spark.table("silver.factpurchaseorder"))

In [0]:
display(
    spark.sql("""
        SELECT
          COUNT(*) AS total_rows,
          COUNT(CurrencyKey) AS rows_with_currency_key,
          COUNT(CostCenterKey) AS rows_with_costcenter_key,
          COUNT(VatAmount) AS rows_with_vat,
          COUNT(TotalAmount) AS rows_with_total_amount,
          COUNT(OrderDateKey) AS rows_with_order_date_key,
          COUNT(ShipDateKey) AS rows_with_ship_date_key,
          COUNT(DeliveredDateKey) AS rows_with_delivered_date_key
        FROM silver.factpurchaseorder
    """)
)

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM silver.factpurchaseorder
        WHERE CurrencyKey IS NULL
           OR VatAmount IS NULL
           OR TotalAmount IS NULL
        LIMIT 20
    """)
)